In [3]:
import pypsa
import pandas as pd
from pathlib import Path

n = pypsa.Network(
    Path.cwd().parent / 'results' / 'networks' / 'base_s_50__168H-T-H-B-I-A-dist1_2035_NT_3000.nc'
)

INFO:pypsa.io:Imported network base_s_50__168H-T-H-B-I-A-dist1_2035_NT_3000.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores


In [4]:
g = n.generators.index[n.generators.carrier == 'gas']
n.generators_t.p[g].sum(axis=1).mul(n.snapshot_weightings.generators, axis=0).sum() / 1e6 * n.snapshot_weightings.generators.sum() / (len(n.snapshots) * 168)

2462.571778237055

In [5]:
idx = pd.IndexSlice
eb = n.statistics.energy_balance().loc[idx[:,['gas for industry', 'gas for industry CC', 'SMR', 'SMR CC'],:]]

In [6]:
eb.sort_values() / 1e6

component  carrier              bus_carrier     
Load       gas for industry     gas for industry   -468.920000
Link       gas for industry     gas                -468.907620
           SMR CC               gas                -188.647911
           SMR                  gas                 -76.308514
           gas for industry CC  gas                  -0.013755
                                co2                   0.000272
                                co2 stored            0.002451
                                gas for industry      0.012380
           SMR CC               co2                   3.735229
           SMR                  co2                  15.109086
           SMR CC               co2 stored           33.617058
           SMR                  Hydrogen Storage     57.994471
           gas for industry     co2                  92.843709
           SMR CC               Hydrogen Storage    130.167059
           gas for industry     gas for industry    468.907620
dtype:

In [7]:
n.statistics.energy_balance().loc[idx[:, :, 'gas for industry']]

component  carrier            
Link       gas for industry       4.689076e+08
           gas for industry CC    1.237960e+04
Load       gas for industry      -4.689200e+08
dtype: float64

In [8]:
ratios = pd.read_csv(
    Path.cwd().parent / 'resources' / 'industrial_energy_demand_base_s_50_2035.csv', header=[0, 1], index_col=0
)

In [9]:
ratios.loc[:, idx['exogenous', ['methane']]].head()

,exogenous
,methane
TWh/a (MtCO2/a),
AL0 0,0.42
AT0 0,21.86
BA0 0,2.30
BE0 0,35.32
BG0 0,6.78


In [10]:
ratios.loc[:, idx['endogenous', ['methane', 'low-temperature heat', 'heat100-200', 'heat200-500', 'heat>500']]].head()

endogenous                                               \
                   methane low-temperature heat heat100-200 heat200-500   
TWh/a (MtCO2/a)                                                           
AL0 0                 0.30                 0.27        0.22        0.07   
AT0 0                13.50                 5.12       13.76        2.85   
BA0 0                 2.21                 0.46        0.45        0.19   
BE0 0                16.65                11.66       14.26        4.17   
BG0 0                 3.34                 1.97        2.63        1.03   

                          
                heat>500  
TWh/a (MtCO2/a)           
AL0 0               0.08  
AT0 0               8.12  
BA0 0               0.74  
BE0 0              14.73  
BG0 0               4.35

In [11]:
cons = [1500, 2000, 2500, 3000, 3500]
template = 'base_s_50__168H-T-H-B-I-A-dist1_2035_NT_{}.nc'

caps = []

for con in cons[::-1]:

    n = pypsa.Network(
        Path.cwd().parent / 'results' / 'networks' / template.format(con)
    )

    caps.append(
        n.statistics.optimal_capacity().rename(con)
    )

caps = pd.concat(caps, axis=1)

INFO:pypsa.io:Imported network base_s_50__168H-T-H-B-I-A-dist1_2035_NT_3500.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores
INFO:pypsa.io:Imported network base_s_50__168H-T-H-B-I-A-dist1_2035_NT_3000.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores
INFO:pypsa.io:Imported network base_s_50__168H-T-H-B-I-A-dist1_2035_NT_2500.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores
INFO:pypsa.io:Imported network base_s_50__168H-T-H-B-I-A-dist1_2035_NT_2000.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores
INFO:pypsa.io:Imported network base_s_50__168H-T-H-B-I-A-dist1_2035_NT_1500.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores


In [12]:
ls = caps.loc[idx['Link']] / 1e3
ls.loc[ls.index.str.contains('urban central')].astype(int)

,3500,3000,2500,2000,1500
carrier,,,,,
urban central air heat pump,26,26,26,37,35
urban central gas CHP,145,145,145,223,240
urban central gas CHP CC,0,0,0,0,0
urban central gas boiler,126,126,126,43,13
urban central solid biomass CHP,22,22,22,48,81
urban central solid biomass CHP CC,0,0,0,0,0


In [15]:
n.links.loc[n.links.carrier == 'urban central gas boiler', ['p_nom', 'p_nom_opt', 'p_nom_extendable', 'p_nom_max', 'p_nom_min']].head()

,p_nom,p_nom_opt,p_nom_extendable,p_nom_max,p_nom_min
Link,,,,,
AL0 0 urban central gas boiler,0.000000,47.553699,True,inf,0.0
AT0 0 urban central gas boiler,513.965453,0.000030,True,inf,0.0
BA0 0 urban central gas boiler,0.000000,318.072351,True,inf,0.0
BE0 0 urban central gas boiler,2905.959938,0.000094,True,inf,0.0
BG0 0 urban central gas boiler,13.023643,0.000019,True,inf,0.0


In [16]:
n.loads.carrier.unique()

array(['electricity', 'land transport EV', 'land transport oil',
       'urban central heat', 'heat100-200 industry',
       'heat200-500 industry', 'heat>500 industry',
       'solid biomass for industry', 'gas for industry',
       'H2 for industry', 'industry methanol', 'naphtha for industry',
       'low-temperature heat for industry', 'industry electricity',
       'process emissions', 'NH3', 'coal for industry',
       'shipping methanol', 'shipping oil', 'kerosene for aviation',
       'agriculture electricity', 'agriculture heat',
       'agriculture machinery electric', 'agriculture machinery oil',
       'rural heat', 'urban decentral heat'], dtype=object)

In [ ]:
n.loads.loc[n.loads.carrier == 'urban ']

In [18]:
eu27_countries = [
    "AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR", "HU",
    "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT", "RO", "SK", "SI", "ES", "SE",
]

In [22]:
noneu = n.buses.index.str[:2].unique().difference(eu27_countries).drop(['EU', 'co'])

In [35]:
idx = pd.IndexSlice
w = n.statistics.withdrawal(groupby=['bus', 'bus_carrier']).loc[idx['Load']]

(w.groupby(w.index.get_level_values(0).str[:2]).sum().loc[noneu] / 1e6).astype(int).sort_values()

# w = w.loc[w.index.get_level_values(0).str[:2].isin(noneu)]

Bus
ME       6
XK      10
AL      12
MK      16
BA      33
RS      78
CH     182
NO     229
GB    1149
dtype: int64

In [38]:
# values taken from FES 2025, Data Workbook, Sheet F.20, (optimistic) Holistic Transition Scenario
uk_gas_consumption = {
    2030: {
        'Power Generation': 59.1, # TWh/a
        'Residential&Tertiary': 304.6, # TWh/a
        'Industry': 158.7, # TWh/a
    },
    2035: {
        'Power Generation': 58.8, # TWh/a
        'Residential&Tertiary': 231.9, # TWh/a
        'Industry': 113.8, # TWh/a
    },
}

# Norway; values taken from Energy Transition Outlook 2024, Page 33
# https://www.norskindustri.no/siteassets/dokumenter/rapporter-og-brosjyrer/energy-transition-norway/energy-transition-norway-2024.pdf
no_gas_consumption = {
    2030: {
        'Power Generation': 0., # TWh/a (approximately zero)
        'Residential&Tertiary': 0., # TWh/a (approximately zero, dominated by electric and biomass heating)
        'Industry': 250 * 0.278, # TWh/a (0.278 is conversion from PJ to TWh)
    },
    2035: {
        'Power Generation': 0., # TWh/a (approximately zero)
        'Residential&Tertiary': 0., # TWh/a (approximately zero, dominated by electric and biomass heating)
        'Industry': 300 * 0.278, # TWh/a (0.278 is conversion from PJ to TWh)
    },
}

# Switzerland; from Energieperspektiven 2050+, Tabelle 62
# see https://www.bfe.admin.ch/bfe/de/home/politik/energieperspektiven-2050-plus.html
ch_gas_consumption = {
    2030: {
        'Power Generation': 0., # TWh/a
        'Residential&Tertiary': 12., # TWh/a
        'Industry': 12., # TWh/a
    },
    2035: {
        'Power Generation': 0., # TWh/a (approximately zero)
        'Residential&Tertiary': 9., # TWh/a
        'Industry': 90., # TWh/a
    },
}

# Combine into a single DataFrame
data = []
for country, consumption_dict in [('UK', uk_gas_consumption), ('NO', no_gas_consumption), ('CH', ch_gas_consumption)]:
    for year in [2030, 2035]:
        for sector in ['Power Generation', 'Residential&Tertiary', 'Industry']:
            data.append({
                'Country': country,
                'Year': year,
                'Sector': sector,
                'Gas Consumption (TWh/a)': consumption_dict[year][sector]
            })

df = pd.DataFrame(data)
df = df.groupby(['Year', 'Sector'])['Gas Consumption (TWh/a)'].sum()


In [39]:
df

Year  Sector              
2030  Industry                240.2
      Power Generation         59.1
      Residential&Tertiary    316.6
2035  Industry                287.2
      Power Generation         58.8
      Residential&Tertiary    240.9
Name: Gas Consumption (TWh/a), dtype: float64